# Generalized DBSCAN clustering

Clusters all 445 feature titles together, without filtering by product type.

Like the other clustering notebooks, this one loads the shared similarity matrix from `data/similarity.npz` (saved by `3_similarity.ipynb`). This keeps the input fixed, so result differences come from the algorithm itself.

**Method**:
1. Convert similarity to distance (`distance = 1 - similarity`).
2. Try a compact grid of DBSCAN parameters (`eps`, `min_samples`).
3. Select a practical configuration based on silhouette and noise ratio.
4. Inspect clusters and visualize with t-SNE.

# Reason for doing DBSCAN 

You don’t have to choose the number of clusters in advance. DBSCAN discovers dense groups from the pairwise distances, unlike methods that need a target cluster count.

It can leave uncertain titles unclustered. Noise points are useful if your goal is to identify only convincing groups of similar features rather than force every title into a group.

It can find irregularly shaped groups and does not rely on clusters being spherical, as k-means does.

It makes a useful comparison with agglomerative and spectral clustering: all use the shared similarity data, but DBSCAN groups items according to local density.


In [171]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

EPS_VALUES = np.arange(0.20, 0.7, 0.02)
MIN_SAMPLES_VALUES = [2, 3, 4, 5, 6, 8, 10, 12, 14]
MIN_COVERAGE = 0.6 #Requirement for at least 50% of titles to be clustered.
MIN_CLUSTER_SIZE = 2
SEED = 0
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    attributes = data["titles"].tolist()

n = len(attributes)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"

distance_matrix = 1 - similarity
distance_matrix = np.clip(distance_matrix, 0, 1)
np.fill_diagonal(distance_matrix, 0)

print(f"Loaded {n} attributes, matrix {similarity.shape}")

Loaded 445 attributes, matrix (445, 445)


In [172]:
def mark_small_clusters_as_noise(labels, min_cluster_size=MIN_CLUSTER_SIZE):
    labels = np.asarray(labels).copy()

    for label in np.unique(labels):
        if label == -1:
            continue
        if np.count_nonzero(labels == label) < min_cluster_size:
            labels[labels == label] = -1

    return labels

## 1. Parameter search

For each `(eps, min_samples)` pair, we fit DBSCAN on the precomputed distance matrix.

We keep only runs with at least 2 non-noise clusters, then compare:
- silhouette score (higher is better),
- noise ratio (lower is better, used as tie-breaker).

In [173]:
def evaluate_dbscan(labels, D):
    non_noise_mask = labels != -1
    n_noise = int((~non_noise_mask).sum())
    noise_ratio = n_noise / len(labels)

    non_noise_labels = labels[non_noise_mask]
    unique_clusters = sorted(set(non_noise_labels.tolist()))
    n_clusters = len(unique_clusters)

    if n_clusters < 2:
        return None

    reduced_distances = D[np.ix_(non_noise_mask, non_noise_mask)]
    silhouette = silhouette_score(reduced_distances, non_noise_labels, metric="precomputed")

    return {
        "silhouette": float(silhouette),
        "noise_ratio": float(noise_ratio),
        "n_clusters": int(n_clusters),
        "n_noise": int(n_noise),
    }


candidates = []
for eps in EPS_VALUES:
    for min_samples in MIN_SAMPLES_VALUES:
        model = DBSCAN(eps=float(eps), min_samples=min_samples, metric="precomputed")
        labels = model.fit_predict(distance_matrix)


        #get rid of clusters that are too small
        labels = mark_small_clusters_as_noise(labels)
       

        evaluation = evaluate_dbscan(labels, distance_matrix)
        if evaluation is None:
            continue
        candidates.append(
            {
                "eps": float(eps),
                "min_samples": int(min_samples),
                **evaluation,
            }
        )

if not candidates:
    raise ValueError("No valid DBSCAN configuration produced at least 2 non-noise clusters.")

candidates = sorted(candidates, key=lambda row: (-row["silhouette"], row["noise_ratio"], row["n_clusters"]))
best = candidates[0]

print("Top 10 parameter sets (sorted by silhouette desc, noise asc):")
for row in candidates[:10]:
    print(
        f"eps={row['eps']:.2f}, min_samples={row['min_samples']} | "
        f"silhouette={row['silhouette']:.4f}, clusters={row['n_clusters']}, "
        f"noise={row['n_noise']} ({100 * row['noise_ratio']:.1f}%)"
    )

print("\nSelected configuration:")
print(
    f"eps={best['eps']:.2f}, min_samples={best['min_samples']} | "
    f"silhouette={best['silhouette']:.4f}, clusters={best['n_clusters']}, "
    f"noise={best['n_noise']} ({100 * best['noise_ratio']:.1f}%)"
)



Top 10 parameter sets (sorted by silhouette desc, noise asc):
eps=0.20, min_samples=3 | silhouette=0.8714, clusters=3, noise=436 (98.0%)
eps=0.22, min_samples=3 | silhouette=0.8714, clusters=3, noise=436 (98.0%)
eps=0.20, min_samples=2 | silhouette=0.8470, clusters=15, noise=412 (92.6%)
eps=0.22, min_samples=2 | silhouette=0.8313, clusters=17, noise=408 (91.7%)
eps=0.24, min_samples=3 | silhouette=0.8017, clusters=4, noise=432 (97.1%)
eps=0.26, min_samples=3 | silhouette=0.8017, clusters=4, noise=432 (97.1%)
eps=0.28, min_samples=3 | silhouette=0.8017, clusters=4, noise=432 (97.1%)
eps=0.30, min_samples=3 | silhouette=0.8017, clusters=4, noise=432 (97.1%)
eps=0.32, min_samples=3 | silhouette=0.8017, clusters=4, noise=432 (97.1%)
eps=0.34, min_samples=3 | silhouette=0.8017, clusters=4, noise=432 (97.1%)

Selected configuration:
eps=0.20, min_samples=3 | silhouette=0.8714, clusters=3, noise=436 (98.0%)


In [174]:
eligible_candidates = [
    row for row in candidates
    if 1.0 - row["noise_ratio"] >= MIN_COVERAGE
]

if not eligible_candidates:
    best_coverage = max(1.0 - row["noise_ratio"] for row in candidates)
    raise ValueError(
        f"No configuration reached minimum coverage of {MIN_COVERAGE:.0%}. "
        f"Best coverage in this search: {best_coverage:.1%}."
    )

ranked_candidates = sorted(
    eligible_candidates,
    key=lambda row: (-row["silhouette"], row["noise_ratio"], row["n_clusters"]),
)
best = ranked_candidates[0]

print(
    f"Top 10 parameter sets meeting minimum coverage ({MIN_COVERAGE:.0%}), "
    "sorted by silhouette desc, noise asc:"
)
for row in ranked_candidates[:10]:
    coverage = 1.0 - row["noise_ratio"]
    print(
        f"eps={row['eps']:.2f}, min_samples={row['min_samples']} | "
        f"silhouette={row['silhouette']:.4f}, clusters={row['n_clusters']}, "
        f"coverage={coverage:.1%}, noise={row['n_noise']} "
        f"({row['noise_ratio']:.1%})"
    )

print("\nSelected configuration:")
print(
    f"eps={best['eps']:.2f}, min_samples={best['min_samples']} | "
    f"silhouette={best['silhouette']:.4f}, clusters={best['n_clusters']}, "
    f"coverage={1.0 - best['noise_ratio']:.1%}, "
    f"noise={best['n_noise']} ({best['noise_ratio']:.1%})"
)

Top 10 parameter sets meeting minimum coverage (60%), sorted by silhouette desc, noise asc:
eps=0.64, min_samples=2 | silhouette=0.1497, clusters=53, coverage=67.0%, noise=147 (33.0%)
eps=0.66, min_samples=2 | silhouette=0.1042, clusters=44, coverage=71.5%, noise=127 (28.5%)
eps=0.66, min_samples=3 | silhouette=0.0879, clusters=21, coverage=61.1%, noise=173 (38.9%)
eps=0.68, min_samples=2 | silhouette=0.0744, clusters=37, coverage=76.9%, noise=103 (23.1%)
eps=0.68, min_samples=3 | silhouette=0.0661, clusters=20, coverage=69.2%, noise=137 (30.8%)

Selected configuration:
eps=0.64, min_samples=2 | silhouette=0.1497, clusters=53, coverage=67.0%, noise=147 (33.0%)


Top 10 parameter sets meeting minimum coverage (70%), sorted by silhouette desc, noise asc:
eps=0.56, min_samples=2 | silhouette=0.1446, clusters=50, coverage=73.0%, noise=120 (27.0%)
eps=0.58, min_samples=2 | silhouette=0.1117, clusters=46, coverage=76.2%, noise=106 (23.8%)
eps=0.60, min_samples=2 | silhouette=0.0939, clusters=35, coverage=80.9%, noise=85 (19.1%)
eps=0.60, min_samples=3 | silhouette=0.0853, clusters=23, coverage=75.5%, noise=109 (24.5%)
eps=0.62, min_samples=4 | silhouette=0.0692, clusters=12, coverage=72.1%, noise=124 (27.9%)
eps=0.64, min_samples=5 | silhouette=0.0498, clusters=7, coverage=73.3%, noise=119 (26.7%)
eps=0.64, min_samples=4 | silhouette=0.0478, clusters=7, coverage=77.8%, noise=99 (22.2%)
eps=0.68, min_samples=8 | silhouette=0.0474, clusters=2, coverage=73.7%, noise=117 (26.3%)
eps=0.68, min_samples=4 | silhouette=0.0459, clusters=4, coverage=86.3%, noise=61 (13.7%)
eps=0.66, min_samples=5 | silhouette=0.0405, clusters=6, coverage=80.2%, noise=88 (19.8%)

Selected configuration:
eps=0.56, min_samples=2 | silhouette=0.1446, clusters=50, coverage=73.0%, noise=120 (27.0%)
The results of weights: 0.5, 0.25, 0.25

Top 10 parameter sets meeting minimum coverage (70%), sorted by silhouette desc, noise asc:

Selected configuration:
eps=0.56, min_samples=2 | silhouette=0.1, clusters=50, coverage=73.0%, noise=120 (27.0%)
The results of weights: 0.7, 0.15, 0.15

Top 10 parameter sets meeting minimum coverage (70%), sorted by silhouette desc, noise asc:
eps=0.66, min_samples=2 | silhouette=0.1042, clusters=44, coverage=71.5%, noise=127 (28.5%)
eps=0.68, min_samples=2 | silhouette=0.0744, clusters=37, coverage=76.9%, noise=103 (23.1%)

Selected configuration:
eps=0.66, min_samples=2 | silhouette=0.1042, clusters=44, coverage=71.5%, noise=127 (28.5%)
The results of weights: 0.25, 0.5, 0.25

Top 10 parameter sets meeting minimum coverage (70%), sorted by silhouette desc, noise asc:
eps=0.68, min_samples=2 | silhouette=0.0873, clusters=41, coverage=74.4%, noise=114 (25.6%)

Selected configuration:
eps=0.68, min_samples=2 | silhouette=0.0873, clusters=41, coverage=74.4%, noise=114 (25.6%)
giving 75% to decriptions and 25 % to titles


## 2. Fit DBSCAN and inspect groups

In [175]:
final_model = DBSCAN(eps=best["eps"], min_samples=best["min_samples"], metric="precomputed")
labels = final_model.fit_predict(distance_matrix)

#
labels = mark_small_clusters_as_noise(labels)

is_noise = labels == -1
cluster_labels = [label for label in sorted(set(labels.tolist())) if label != -1]
sizes = Counter(labels)

print(
    f"Clusters (excluding noise): {len(cluster_labels)} | "
    f"noise points: {sizes.get(-1, 0)} / {len(labels)} ({100 * sizes.get(-1, 0) / len(labels):.1f}%)"
)
print(f"Cluster sizes (largest first): {[size for label, size in sizes.most_common() if label != -1]}")

members_by_cluster = {
    label: [attributes[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in cluster_labels
}

for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    if len(members) > 1:
        print(f"\nCluster {label} ({len(members)} attributes)")
        for member in members:
            print(f"  - {member}")

Clusters (excluding noise): 53 | noise points: 147 / 445 (33.0%)
Cluster sizes (largest first): [107, 13, 13, 11, 10, 9, 8, 7, 6, 5, 5, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Cluster 0 (107 attributes)
  - Light insulation
  - Windproof Front Panel
  - Insulating
  - Waterproof and breathable
  - Waterproof contoured lower leg seams
  - Insulating fleece inner
  - Full waterproof protection
  - Durable for daily use
  - Lightweight and fast drying
  - Sweat management
  - Highly breathable
  - Windproof protection
  - Breathable mesh back
  - 3mm neoprene
  - Insulating & breathable
  - Moisture wicking
  - COOLMAX® fibre
  - Fast-drying material
  - Stretchy and waterproof
  - Neoprene wet-weather warmth
  - Naturally insulating and soft
  - Breathable and moisture wicking
  - Merino wool fabric mix
  - Breathable
  - Sweat-wicking
  - Waterproof and breathable insert
  - Coated 3 mm neoprene
  - D

## 3. Clustering evaluation

DBSCAN marks some points as noise (`-1`).

Silhouette is computed only on non-noise points and their pairwise distances.

In [176]:
non_noise_mask = labels != -1
non_noise_labels = labels[non_noise_mask]

print(f"Non-noise points: {int(non_noise_mask.sum())}")
print(f"Noise points: {int((~non_noise_mask).sum())}")
print(f"Clusters (excluding noise): {len(set(non_noise_labels.tolist()))}")

if len(set(non_noise_labels.tolist())) >= 2:
    reduced_distances = distance_matrix[np.ix_(non_noise_mask, non_noise_mask)]
    sil = silhouette_score(reduced_distances, non_noise_labels, metric="precomputed")
    print(f"Silhouette score (non-noise only): {sil:.4f}")
else:
    print("Silhouette score not defined (need at least 2 non-noise clusters).")

Non-noise points: 298
Noise points: 147
Clusters (excluding noise): 53
Silhouette score (non-noise only): 0.1497


## 4. Cluster visualization

t-SNE projects precomputed distances (`1 - similarity`) to two dimensions for display only.
The largest groups receive highlight colors, smaller groups are grey, and noise points are hollow markers.

In [177]:
INK = "#2b2b2b"
GRAY = "#d7d7d7"
LIGHT_GRAY = "#c9c8c0"
HIGHLIGHT_PALETTE = ["#cc1f1a", "#005f73", "#0a9396", "#ee9b00", "#ae2012", "#6c757d", "#9b2226", "#3a86ff"]

perplexity = min(30, len(attributes) - 1)
coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    random_state=SEED,
    perplexity=perplexity,
).fit_transform(distance_matrix)

multi = [label for label in cluster_labels if sizes[label] > 1]
ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]

def group_name(label):
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return attributes[medoid_index]

group_names = {label: group_name(label) for label in multi}

def make_trace(indices, name, color, custom_group_names, hollow=False, size=9):
    if len(indices) == 0:
        return None
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[attributes[index] for index in indices],
        customdata=[custom_group_names[index] for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )

custom_group_names = []
for label in labels:
    if label == -1:
        custom_group_names.append("Noise")
    else:
        custom_group_names.append(group_names.get(label, f"Cluster {label}"))

noise_indices = np.where(labels == -1)[0]
other_group_indices = np.array(
    [index for index, label in enumerate(labels) if label != -1 and sizes[label] > 1 and label not in highlighted_groups],
    dtype=int,
)

figure = go.Figure()

trace = make_trace(noise_indices, f"Noise ({len(noise_indices)})", LIGHT_GRAY, custom_group_names, hollow=True, size=7)
if trace is not None:
    figure.add_trace(trace)

trace = make_trace(
    other_group_indices,
    f"Other groups ({max(len(ranked_groups) - len(highlighted_groups), 0)})",
    GRAY,
    custom_group_names,
    size=7,
)
if trace is not None:
    figure.add_trace(trace)

for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    trace = make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, custom_group_names, size=10)
    if trace is not None:
        figure.add_trace(trace)

figure.update_layout(
    title=dict(text="Attribute clusters: DBSCAN (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

Your impression is reasonable: for your goal of grouping most or all 445 titles, the results so far suggest agglomerative is the better fit. DBSCAN isn’t guaranteed to outperform it; the algorithms assume different shapes in the data.

DBSCAN looks for uniformly dense neighborhoods using one global eps. Here, changing eps seems to trade one problem for another: stricter settings leave many titles as noise; permissive settings merge groups or create many tiny components. With min_samples=1, every title is automatically a core point, so the 339 clusters and 0% noise don’t show that DBSCAN found meaningful density—they show that the setting stopped requiring a dense neighborhood.

Agglomerative always builds groups for every title. Its current silhouette of 0.1841 is also modest, but it covers all titles in 138 clusters. DBSCAN’s 0.2472 result covers only 61.1%, so it isn’t directly comparable; its full-coverage result at min_samples=1 has a lower silhouette (0.1380) and many more clusters. Silhouette isn’t a complete measure of semantic quality, so inspect group members too.

I would stop trying to tune eps just to make DBSCAN look better. Keep agglomerative as the primary result if your goal is broad grouping, and describe DBSCAN as a comparison that struggled with this data’s density structure. For example:

DBSCAN was tested because it can identify dense groups and leave uncertain titles unassigned. However, the results were sensitive to parameter choices: stricter settings produced substantial noise, while min_samples=1 produced many small components. This suggests the data do not have a clear, uniform density structure for DBSCAN. Agglomerative clustering was more suitable for our goal of grouping all feature titles.